# ChakraModel V5 Strict Verification - True Anti-Fabrication Harness

This notebook copies your chosen dataset subset to the working directory, plants the Anti-Fabrication canaries, and runs the hardened `verify_kaggle.py` script via `harness_core`.

**Before running:**
1. Make sure you have added the `ChakraModel_Kaggle_Code` and `Anti-Fabrication Toolkit` datasets.
2. Ensure you run this using **Save Version -> Save & Run All (Commit)**.

In [1]:
!pip install -q ultralytics timm opencv-python-headless pycocotools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.6 MB/s eta 0:00:00


In [2]:
import os
import sys
import shutil
import glob
import json
import re

###################################################################
# CONFIGURATION
###################################################################
DATASET_ROOT = '/kaggle/input/datasets/gokulrocky/polypdb-polyp-raw/PolypDB/PolypDB/PolypDB_center_wise/BKAI'
VERIFY_DATASET_NAME = "PolypDB_BKAI"
WORKING_DATASET = '/kaggle/working/dataset_copy'

# 1. Clean up old working directory
if os.path.exists(WORKING_DATASET):
    shutil.rmtree(WORKING_DATASET)

# 2. Create the flat structure that the Harness strictly requires
os.makedirs(os.path.join(WORKING_DATASET, 'images'))
os.makedirs(os.path.join(WORKING_DATASET, 'masks'))

print(f"Flattening and copying files from all modalities in {DATASET_ROOT} ...")
image_count = 0
mask_count = 0

# 3. Recursively hunt down every image and mask in FICE, BLI, WLI, LCI
for root, dirs, files in os.walk(DATASET_ROOT):
    for file in files:
        # Ignore random hidden files
        if file.startswith('.') or not file.endswith(('.png', '.jpg', '.tif', '.bmp')):
            continue
            
        src = os.path.join(root, file)
        
        # We prefix the filename with the modality name (e.g. FICE_image1.png) 
        # so files with the same name from different modalities don't overwrite each other!
        modality = os.path.basename(os.path.dirname(root)) 
        new_filename = f"{modality}_{file}"
        
        if 'images' in root.lower():
            dst = os.path.join(WORKING_DATASET, 'images', new_filename)
            shutil.copy2(src, dst)
            image_count += 1
        elif 'masks' in root.lower():
            dst = os.path.join(WORKING_DATASET, 'masks', new_filename)
            shutil.copy2(src, dst)
            mask_count += 1

print(f"✅ Flattening Complete!")
print(f"Copied {image_count} real images and {mask_count} real masks into {WORKING_DATASET}")

Flattening and copying files from all modalities in /kaggle/input/datasets/gokulrocky/polypdb-polyp-raw/PolypDB/PolypDB/PolypDB_center_wise/BKAI ...
✅ Flattening Complete!
Copied 1200 real images and 1200 real masks into /kaggle/working/dataset_copy


In [3]:
import os

def print_directory_tree(startpath, max_depth=4):
    print(f"🗺️ Directory tree for: {startpath}")
    print("=" * 50)
    
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        
        # Stop digging if we go too deep (keeps the output readable)
        if level > max_depth:
            continue
            
        indent = ' ' * 4 * level
        folder_name = os.path.basename(root)
        if folder_name == '':
            folder_name = startpath
            
        print(f"{indent}📂 {folder_name}/")
        
        subindent = ' ' * 4 * (level + 1)
        
        # Print the first 3 files in this folder so we can see what type of data is here
        for f in files[:3]:
            print(f"{subindent}📄 {f}")
            
        if len(files) > 3:
            print(f"{subindent}... and {len(files)-3} more files")

print_directory_tree('/kaggle/input', max_depth=4)

🗺️ Directory tree for: /kaggle/input
📂 input/
    📂 datasets/
        📂 gokulrocky/
            📂 chakramodel-kaggle-code/
                📄 requirements.txt
                📂 src/
                    📄 hybrid_refine.py
                    📄 train_yolo.py
                    📄 train_pranet.py
                    ... and 31 more files
            📂 final-om-evlautation-upload/
                📄 Kaggle_ChakraTransformer_Evaluation.ipynb
                📂 weights/
                    📄 chakra_transformer_best.pth
                    📄 best.pt
                📂 src/
                    📄 hybrid_refine.py
                    📄 train_yolo.py
                    📄 metrics_engine_v2.py
                    ... and 32 more files
            📂 polypdb-polyp-raw/
                📂 PolypDB/
        📂 gokulraj324/
            📂 anti-fabrication-toolkit/
                📄 harness_core.py


In [4]:
%%writefile /kaggle/working/verify_kaggle.py
import os
import sys
import glob
import cv2
import numpy as np
import torch
from pathlib import Path
import hashlib

def md5(fname):
    hash_md5 = hashlib.md5()
    with open(fname, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()

def verify_strict():
    print("=" * 60)
    print("STRICT CHECKPOINT AND ARCHITECTURE VERIFICATION")
    print("=" * 60)
    
    # Identify codebase path
    code_dir = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'chakranet_segmenter.py' in files and 'src' in root:
            code_dir = root
            break
    if not code_dir:
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'chakranet_segmenter.py' in files:
                code_dir = root
                break
    assert code_dir, "FATAL: ChakraModel codebase not found in /kaggle/input"
    sys.path.insert(0, code_dir)
    print(f"[VERIFY] Codebase found at: {code_dir}")
    
    from ultralytics import YOLO
    from chakranet_segmenter import ChakraNet
    from metrics_engine_v2 import MetricsEngineV2

    # BUG 3 FIX: Strict Weight Selection
    yolo_candidates = glob.glob('/kaggle/input/**/best.pt', recursive=True)
    vit_candidates = glob.glob('/kaggle/input/**/chakra_transformer_best.pth', recursive=True)
    
    # Strictly filter yolo_candidates (ignore anything with 'transformer' in the path)
    yolo_candidates = [p for p in yolo_candidates if 'transformer' not in p.lower()]
    
    print(f"YOLO candidates found: {yolo_candidates}")
    print(f"ViT candidates found: {vit_candidates}")
    
    assert len(yolo_candidates) == 1, f"FATAL: Expected exactly 1 YOLO weight best.pt, found {len(yolo_candidates)}!"
    assert len(vit_candidates) == 1, f"FATAL: Expected exactly 1 ViT weight chakra_transformer_best.pth, found {len(vit_candidates)}!"
    
    yolo_path = yolo_candidates[0]
    weight_path = vit_candidates[0]

    print(f"Segmenter Checkpoint: {weight_path}")
    print(f"Segmenter MD5: {md5(weight_path)}")
    print(f"YOLO Checkpoint: {yolo_path}")
    print(f"YOLO MD5: {md5(yolo_path)}")
    print("=" * 60)
    
    nonce = os.environ.get("VERIFY_NONCE", "NO_NONCE")
    print(f"VERIFY_NONCE: {nonce}")
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Executing on device: {device}")
    
    yolo_model = YOLO(yolo_path)
    yolo_model.to(device)
    
    segmenter = ChakraNet(img_size=(384, 384), device=device)
    segmenter.model.to(device)
    
    # Load weights correctly by explicitly stripping 'module.'
    sd = torch.load(weight_path, map_location=device)
    sd_fixed = {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd.items()}
    res = segmenter.model.load_state_dict(sd_fixed, strict=False)
    print(f"ChakraNet Weights Loaded. Missing keys: {len(res.missing_keys)}, Unexpected keys: {len(res.unexpected_keys)}")
    
    total_keys = len(segmenter.model.state_dict())
    if len(res.missing_keys) > (0.1 * total_keys):
        print(f"[FATAL] Major weight mismatch! Missing keys > 10% ({len(res.missing_keys)} missing out of {total_keys}).")
        sys.exit(1)
    
    metrics_engine = MetricsEngineV2()
    
    dataset_name = os.environ.get("VERIFY_DATASET_NAME")
    dataset_root = os.environ.get("VERIFY_DATASET_ROOT")
    
    if not dataset_name or not dataset_root:
        print("[FATAL] VERIFY_DATASET_NAME or VERIFY_DATASET_ROOT environment variable not set.")
        sys.exit(1)
        
    print(f"\n============================================================")
    print(f"VERIFYING DATASET: {dataset_name}")
    print(f"============================================================")
    
    d_root = Path(dataset_root)
    images_dir = d_root / "images"
    masks_dir = d_root / "masks"
    
    if not images_dir.exists():
        print(f"[FATAL] Images directory not found: {images_dir}")
        sys.exit(1)
        
    image_paths = sorted(list(images_dir.glob("*.png")) + list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.tif")))
    print(f"Total images found: {len(image_paths)}")
    
    results = {k: [] for k in ["Dice", "mIoU", "wF-measure", "S-measure", "E-measure", "MAE"]}
    import csv
    csv_file = "/kaggle/working/per_image_metrics_log.csv"
    with open(csv_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Filename", "Dice", "mIoU", "wF-measure", "S-measure", "E-measure", "MAE"])
        
    for i, img_path in enumerate(image_paths):
        mask_path = None
        for ext in [".png", ".jpg", ".tif", ".bmp"]:
            potential_path = masks_dir / (img_path.stem + ext)
            if potential_path.exists():
                mask_path = potential_path
                break
                
        if not mask_path:
            print(f"[WARN] Mask not found for {img_path.name}")
            continue
            
        img_bgr = cv2.imread(str(img_path))
        gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        gt_resized = cv2.resize(gt_mask, (448, 448), interpolation=cv2.INTER_NEAREST)
        
        img_resized = cv2.resize(img_bgr, (448, 448))
        yolo_results = yolo_model(img_resized, verbose=False)[0]
        
        # Soft prob map initialized to 0
        c_prob = np.zeros((448, 448), dtype=np.float32)
        
        if len(yolo_results.boxes) > 0:
            boxes = sorted(yolo_results.boxes, key=lambda b: b.conf[0].item(), reverse=True)
            box = boxes[0].xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = map(int, box)
            
            w_box, h_box = x2 - x1, y2 - y1
            px, py = int(0.25 * w_box), int(0.25 * h_box)
            px1, py1 = max(0, x1 - px), max(0, y1 - py)
            px2, py2 = max(0, min(448, x2 + px)), max(0, min(448, y2 + py))
            
            if px2 > px1 and py2 > py1:
                roi_crop = img_resized[py1:py2, px1:px2]
                
                # BUG 2 FIX: Direct manual inference EXACTLY matching chakranet_segmenter.py
                from utils.transforms import letterbox_pad, unletterbox
                img_rgb = cv2.cvtColor(roi_crop, cv2.COLOR_BGR2RGB)
                img_resized_crop, meta = letterbox_pad(img_rgb, target_size=(384, 384))
                
                img_norm = img_resized_crop.astype(np.float32) / 255.0
                mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
                std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
                img_norm = (img_norm - mean) / std
                img_tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).to(device)
                
                try:
                    with torch.no_grad():
                        logits = segmenter.model(img_tensor)
                        prob = torch.sigmoid(logits)
                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        print("[WARN] Switched to CPU due to CUDA OutOfMemory!")
                        img_tensor_cpu = img_tensor.cpu()
                        segmenter.model.to('cpu')
                        with torch.no_grad():
                            logits = segmenter.model(img_tensor_cpu)
                            prob = torch.sigmoid(logits)
                        segmenter.model.to(device)
                    else:
                        raise
                        
                prob_np = prob.squeeze().cpu().numpy()
                prob_resized = unletterbox(prob_np, meta)
                
                c_prob[py1:py2, px1:px2] = prob_resized
        
        # Explicit binarization ONLY for the ground truth here (network outputs soft prob)
        gt_bin = (gt_resized > 127).astype(np.uint8)
        
        # Pass continuous probability map (c_prob) directly to compute_all 
        # (compute_all handles thresholding for Dice/IoU internally)
        metrics = metrics_engine.compute_all(c_prob, gt_bin)
        
        for k in results.keys():
            results[k].append(metrics[k])
            
        # BUG 4 FIX: Checkpoint saving per image
        with open(csv_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([img_path.name, metrics['Dice'], metrics['mIoU'], metrics['wF-measure'], metrics['S-measure'], metrics['E-measure'], metrics['MAE']])
            
        # IMPORTANT: Print per-image dice WITH FILENAME for canary detection
        dice_score = metrics['Dice']
        print(f"File: {img_path.name} | Dice: {dice_score:.4f}")
        
    print(f"\n[DONE] {dataset_name}")
    print(f"Total Evaluated Images: {len(results['Dice'])}")
    for k in results.keys():
        print(f"Final True Average {k}: {float(np.mean(results[k])):.4f}")
    print(f"============================================================")

if __name__ == "__main__":
    verify_strict()



Writing /kaggle/working/verify_kaggle.py


In [5]:
harness_path = glob.glob('/kaggle/input/**/harness_core.py', recursive=True)
if not harness_path:
    harness_path = glob.glob('/kaggle/working/**/harness_core.py', recursive=True)

if not harness_path:
    print("FATAL: harness_core.py not found! Did you attach the Anti-Fabrication Toolkit dataset?")
    sys.exit(1)

toolkit_dir = os.path.dirname(harness_path[0])
sys.path.insert(0, toolkit_dir)

from harness_core import plant_multi_canary, run_live, verify, sign_verdict

print("Planting canaries...")
secret = plant_multi_canary(WORKING_DATASET)

extra_env = {
    "VERIFY_NONCE": secret["nonce"],
    "VERIFY_DATASET_NAME": VERIFY_DATASET_NAME,
    "VERIFY_DATASET_ROOT": WORKING_DATASET
}

print(f"\nRunning verification for {VERIFY_DATASET_NAME}...")
stdout_text, arrivals, elapsed, retcode = run_live('/kaggle/working/verify_kaggle.py', extra_env, timeout_sec=10000)

print(stdout_text)

if retcode != 0:
    print(f"FATAL: verify_kaggle.py exited with code {retcode}")
else:
    # Parse claimed total from stdout
    match = re.search(r"Total Evaluated Images:\s*(\d+)", stdout_text)
    claimed_total = int(match.group(1)) if match else -1
    
    # Use harness core to verify output logic
    real_count = len(os.listdir(os.path.join(WORKING_DATASET, "images")))
    verdict_dict = verify(stdout_text, arrivals, elapsed, secret, claimed_total, real_count)
    
    card = {
        "project": "ChakraModel",
        "dataset": VERIFY_DATASET_NAME,
        "nonce": secret["nonce"],
        "decision": "ADMITTED" if verdict_dict["pass"] else "WITHHELD",
        "findings": verdict_dict["findings"],
        "elapsed_sec": verdict_dict["elapsed_sec"]
    }
    
    signed_card = sign_verdict(card, secret["hmac_key"])
    
    with open('/kaggle/working/completion_card.json', 'w') as f:
        json.dump(signed_card, f, indent=2)
        
    print(f"\n=========================================")
    print(f"VERDICT: {card['decision']}")
    for finding in card['findings']:
        print(f"  - {finding}")
    print(f"Saved to completion_card.json")
    print(f"=========================================")


Planting canaries...

Running verification for PolypDB_BKAI...
STRICT CHECKPOINT AND ARCHITECTURE VERIFICATION
[VERIFY] Codebase found at: /kaggle/input/datasets/gokulrocky/chakramodel-kaggle-code/src
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
[WARN] HardwareMonitor unavailable: [Errno 30] Read-only file system: '/kaggle/input/datasets/gokulrocky/chakramodel-kaggle-code/logs'
Traceback (most recent call last):
  File "/kaggle/working/verify_kaggle.py", line 210, in <module>
    verify_strict()
  File "/kaggle/working/verify_kaggle.py", line 39, in verify_strict
    from metrics_engine_v2 import MetricsEngineV2
ModuleNotFoundError: No module named 'metrics_engine_v2'

FATAL: verify_kaggle.py exited with code 1


In [6]:
import os
bkai_path = '/kaggle/input/datasets/gokulrocky/polypdb-polyp-raw/PolypDB/PolypDB/PolypDB_center_wise/BKAI'

print(f"Contents of BKAI:")
items = os.listdir(bkai_path)
for item in items[:20]:
    path = os.path.join(bkai_path, item)
    if os.path.isdir(path):
        print(f"📂 [FOLDER] {item}")
    else:
        print(f"📄 [FILE]   {item}")
        
if len(items) > 20:
    print(f"... and {len(items)-20} more items.")

Contents of BKAI:
📂 [FOLDER] FICE
📂 [FOLDER] BLI
📂 [FOLDER] WLI
📂 [FOLDER] LCI


In [7]:
!pip install ultralytics

In [8]:
import os
import sys
import glob
import cv2
import numpy as np
import torch
from pathlib import Path

# ---------------------------------------------------------------------------
# Locate ChakraModel codebase
# ---------------------------------------------------------------------------
# Add all src directories in Kaggle datasets to sys.path to ensure we find everything
src_dirs_added = []
for root, dirs, files in os.walk('/kaggle/input'):
    if os.path.basename(root) == 'src':
        sys.path.insert(0, root)
        src_dirs_added.append(root)

if not src_dirs_added:
    # Fallback just in case
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'chakranet_segmenter.py' in files:
            sys.path.insert(0, root)
            src_dirs_added.append(root)

print(f"[REPRODUCIBILITY] Added {len(src_dirs_added)} codebase directories to sys.path:")
for d in src_dirs_added:
    print(f"  -> {d}")

from ultralytics import YOLO
from chakranet_segmenter import ChakraNet
from metrics_engine_v2 import MetricsEngineV2


def compute_iou(box1, box2):
    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    iou = intersection_area / float(box1_area + box2_area - intersection_area + 1e-6)
    return iou


def get_gt_bbox(mask_img):
    coords = cv2.findNonZero(mask_img)
    if coords is None:
        return None
    x, y, w, h = cv2.boundingRect(coords)
    return [x, y, x + w, y + h]


def run_diagnostics():
    print("=" * 60)
    print("STARTING DIAGNOSTIC EVALUATION (YOLO vs ChakraNet)")
    print("=" * 60)

    # -----------------------------------------------------------------------
    # Locate weights - FIX #4: print exactly what's chosen, don't pick silently
    # -----------------------------------------------------------------------
    yolo_candidates = sorted(glob.glob('/kaggle/input/**/best.pt', recursive=True))
    vit_candidates = sorted(glob.glob('/kaggle/input/**/chakra_transformer_best.pth', recursive=True))

    if not yolo_candidates or not vit_candidates:
        print("FATAL: could not find YOLO best.pt or chakra_transformer_best.pth")
        sys.exit(1)

    if len(yolo_candidates) > 1:
        print(f"[WARNING] Multiple YOLO checkpoints found: {yolo_candidates}")
    if len(vit_candidates) > 1:
        print(f"[WARNING] Multiple ChakraNet checkpoints found: {vit_candidates}")

    yolo_path = yolo_candidates[-1]
    weight_path = vit_candidates[-1]
    print(f"[REPRODUCIBILITY] YOLO checkpoint : {yolo_path}")
    print(f"[REPRODUCIBILITY] ChakraNet ckpt  : {weight_path}")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[INFO] Device: {device}")

    # Load YOLO
    yolo_model = YOLO(yolo_path)
    yolo_model.to(device)

    # -----------------------------------------------------------------------
    # Load ChakraNet - FIX #2: strict loading with explicit key reporting
    # -----------------------------------------------------------------------
    segmenter = ChakraNet(img_size=(384, 384), device=device)
    segmenter.model.to(device)
    sd = torch.load(weight_path, map_location=device)
    sd_fixed = {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd.items()}

    load_result = segmenter.model.load_state_dict(sd_fixed, strict=False)
    if load_result.missing_keys:
        print(f"[CHECKPOINT WARNING] {len(load_result.missing_keys)} missing keys:")
        for k in load_result.missing_keys[:20]:
            print(f"    MISSING: {k}")
        if len(load_result.missing_keys) > 20:
            print(f"    ... and {len(load_result.missing_keys) - 20} more")
    if load_result.unexpected_keys:
        print(f"[CHECKPOINT WARNING] {len(load_result.unexpected_keys)} unexpected keys:")
        for k in load_result.unexpected_keys[:20]:
            print(f"    UNEXPECTED: {k}")
        if len(load_result.unexpected_keys) > 20:
            print(f"    ... and {len(load_result.unexpected_keys) - 20} more")
    if not load_result.missing_keys and not load_result.unexpected_keys:
        print("[CHECKPOINT OK] All keys matched exactly.")
    else:
        print("[CHECKPOINT WARNING] Mismatches above mean results may not reflect "
              "your intended trained weights. Investigate before trusting metrics below.")

    # FIX #1: eval mode - THIS WAS MISSING IN THE ORIGINAL DRAFT
    segmenter.model.eval()
    print("[FIX APPLIED] segmenter.model.eval() called - BN/Dropout now in inference mode.")

    metrics_engine = MetricsEngineV2()

    images_dir = Path("/kaggle/working/dataset_copy/images")
    masks_dir = Path("/kaggle/working/dataset_copy/masks")

    image_paths = sorted(list(images_dir.glob("*.png")) + list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.tif")))
    image_paths = [p for p in image_paths if not p.name.startswith("CANARY")]

    print(f"Total valid diagnostic images: {len(image_paths)}")
    print("[ASSUMPTION CHECK] Oracle crop uses 25% bbox padding, assumed to match "
          "your E2E pipeline's YOLO->crop expansion logic. If your actual pipeline "
          "uses a different padding ratio, the oracle number below is not apples-to-apples "
          "with your full-pipeline Dice of 0.8370 - verify this before drawing conclusions.")

    yolo_ious = []
    yolo_hits_any = 0       # IoU > 0.0
    yolo_hits_50 = 0        # IoU >= 0.5  (FIX #3)
    yolo_misses = 0

    chakra_results = {k: [] for k in ["Dice", "mIoU", "wF-measure", "S-measure", "E-measure", "MAE"]}
    per_image_dice = []     # FIX #5

    for i, img_path in enumerate(image_paths):
        mask_path = None
        for ext in [".png", ".jpg", ".tif", ".bmp"]:
            potential = masks_dir / (img_path.stem + ext)
            if potential.exists():
                mask_path = potential
                break

        if not mask_path:
            continue

        img_bgr = cv2.imread(str(img_path))
        gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        img_resized = cv2.resize(img_bgr, (448, 448))
        gt_resized = cv2.resize(gt_mask, (448, 448), interpolation=cv2.INTER_NEAREST)
        gt_bin = (gt_resized > 127).astype(np.uint8)

        gt_bbox = get_gt_bbox(gt_bin)

        if gt_bbox is None:
            continue

        # ---------------------------------------------------------------
        # 1. DIAGNOSTIC: YOLO ONLY (Detection)
        # ---------------------------------------------------------------
        yolo_results = yolo_model(img_resized, verbose=False)[0]
        if len(yolo_results.boxes) > 0:
            boxes = sorted(yolo_results.boxes, key=lambda b: b.conf[0].item(), reverse=True)
            pred_box = boxes[0].xyxy[0].cpu().numpy().tolist()

            iou = compute_iou(gt_bbox, pred_box)
            yolo_ious.append(iou)
            if iou > 0.0:
                yolo_hits_any += 1
            else:
                yolo_misses += 1
            if iou >= 0.5:
                yolo_hits_50 += 1
        else:
            yolo_ious.append(0.0)
            yolo_misses += 1

        # ---------------------------------------------------------------
        # 2. DIAGNOSTIC: CHAKRANET ONLY (Perfect Crops)
        # ---------------------------------------------------------------
        x1, y1, x2, y2 = gt_bbox
        w_box, h_box = x2 - x1, y2 - y1

        px, py = int(0.25 * w_box), int(0.25 * h_box)
        px1, py1 = max(0, x1 - px), max(0, y1 - py)
        px2, py2 = max(0, min(448, x2 + px)), max(0, min(448, y2 + py))

        if px2 > px1 and py2 > py1:
            roi_crop = img_resized[py1:py2, px1:px2]

            img_rgb = cv2.cvtColor(roi_crop, cv2.COLOR_BGR2RGB)
            img_resized_crop = cv2.resize(img_rgb, (384, 384))
            img_norm = img_resized_crop.astype(np.float32) / 255.0
            mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
            std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
            img_norm = (img_norm - mean) / std
            img_tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).to(device)

            with torch.no_grad():
                logits = segmenter.model(img_tensor)
                prob = torch.sigmoid(logits)

            prob_np = prob.squeeze().cpu().numpy()
            prob_resized = cv2.resize(prob_np, (px2 - px1, py2 - py1), interpolation=cv2.INTER_LINEAR)

            c_prob = np.zeros((448, 448), dtype=np.float32)
            c_prob[py1:py2, px1:px2] = prob_resized

            metrics = metrics_engine.compute_all(c_prob, gt_bin)
            for k in chakra_results.keys():
                chakra_results[k].append(metrics[k])
            per_image_dice.append(metrics["Dice"])

        if i % 100 == 0:
            print(f"Processed {i}/{len(image_paths)} images...")

    print("\n=========================================")
    print("YOLO DETECTION PERFORMANCE (DIAGNOSTIC)")
    print("=========================================")
    print(f"Average Bounding Box IoU: {np.mean(yolo_ious):.4f}")
    total_yolo = yolo_hits_any + yolo_misses
    print(f"Detection Hit Rate (IoU > 0.0):  {yolo_hits_any}/{total_yolo} ({yolo_hits_any/total_yolo*100:.2f}%)")
    print(f"Detection Recall  (IoU >= 0.5):  {yolo_hits_50}/{total_yolo} ({yolo_hits_50/total_yolo*100:.2f}%)")

    print("\n=========================================")
    print("CHAKRANET SEGMENTATION PERFORMANCE (PERFECT CROP)")
    print("=========================================")
    for k in chakra_results.keys():
        print(f"Perfect Crop Average {k}: {np.mean(chakra_results[k]):.4f}")

    if per_image_dice:
        arr = np.array(per_image_dice)
        print("\n=========================================")
        print("PER-IMAGE DICE DISTRIBUTION (bimodal check)")
        print("=========================================")
        bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.01]
        labels = ["0.0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"]
        counts, _ = np.histogram(arr, bins=bins)
        for lbl, c in zip(labels, counts):
            print(f"  Dice {lbl}: {c} images ({c/len(arr)*100:.1f}%)")
        print(f"  Median Dice: {np.median(arr):.4f}  |  Std Dice: {np.std(arr):.4f}")
        print("  -> If 0.0-0.2 bucket is large relative to a strong 0.8-1.0 peak, "
              "that's a bimodal signal -> likely a DETECTION/localization problem "
              "(some images fail hard), not smooth segmentation weakness.")
    print("=========================================")


if __name__ == "__main__":
    run_diagnostics()

[REPRODUCIBILITY] Added 2 codebase directories to sys.path:
  -> /kaggle/input/datasets/gokulrocky/chakramodel-kaggle-code/src
  -> /kaggle/input/datasets/gokulrocky/final-om-evlautation-upload/src
[WARN] HardwareMonitor unavailable: [Errno 30] Read-only file system: '/kaggle/input/datasets/gokulrocky/final-om-evlautation-upload/logs'
STARTING DIAGNOSTIC EVALUATION (YOLO vs ChakraNet)
[REPRODUCIBILITY] YOLO checkpoint : /kaggle/input/datasets/gokulrocky/final-om-evlautation-upload/weights/best.pt
[REPRODUCIBILITY] ChakraNet ckpt  : /kaggle/input/datasets/gokulrocky/final-om-evlautation-upload/weights/chakra_transformer_best.pth
[INFO] Device: cuda
2026-09-06 12:47:16,784 [HW-MONITOR] INFO Loading pretrained weights from Hugging Face hub (timm/vit_large_patch16_384.augreg_in21k_ft_in1k)
2026-09-06 12:47:16,929 [HW-MONITOR] INFO HTTP Request: HEAD https://huggingface.co/timm/vit_large_patch16_384.augreg_in21k_ft_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"


2026-09-06 12:47:16,930 [HW-MONITOR] WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-09-06 12:47:17,028 [HW-MONITOR] INFO HTTP Request: GET https://huggingface.co/api/models/timm/vit_large_patch16_384.augreg_in21k_ft_in1k/xet-read-token/1a0edc93ef27935b955ce67a344087b140f85d47 "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

2026-09-06 12:47:24,231 [HW-MONITOR] INFO [timm/vit_large_patch16_384.augreg_in21k_ft_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
[INFO] ChakraNet: Loaded weights from /kaggle/input/datasets/gokulrocky/final-om-evlautation-upload/weights/chakra_transformer_best.pth
[CHECKPOINT OK] All keys matched exactly.
[FIX APPLIED] segmenter.model.eval() called - BN/Dropout now in inference mode.
Total valid diagnostic images: 1200
[ASSUMPTION CHECK] Oracle crop uses 25% bbox padding, assumed to match your E2E pipeline's YOLO->crop expansion logic. If your actual pipeline uses a different padding ratio, the oracle number below is not apples-to-apples with your full-pipeline Dice of 0.8370 - verify this before drawing conclusions.


error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/resize.cpp:4086: error: (-215:Assertion failed) func != 0 in function 'resize'


In [ ]:
{
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.12.13",
   "mimetype": "text/x-python",
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "pygments_lexer": "ipython3",
   "nbconvert_exporter": "python",
   "file_extension": ".py"
  }
 },
 "nbformat_minor": 4,
 "nbformat": 4,
 "cells": [
  {
   "cell_type": "markdown",
   "source": "# ChakraModel V5 Strict Verification - True Anti-Fabrication Harness\n\nThis notebook copies your chosen dataset subset to the working directory, plants the Anti-Fabrication canaries, and runs the hardened `verify_kaggle.py` script via `harness_core`.\n\n**Before running:**\n1. Make sure you have added the `ChakraModel_Kaggle_Code` and `Anti-Fabrication Toolkit` datasets.\n2. Ensure you run this using **Save Version -> Save & Run All (Commit)**.",
   "metadata": {}
  },
  {
   "cell_type": "code",
   "source": "!pip install -q ultralytics timm opencv-python-headless pycocotools",
   "metadata": {
    "trusted": true,
    "execution": {
     "iopub.status.busy": "2026-09-06T12:32:08.279801Z",
     "iopub.execute_input": "2026-09-06T12:32:08.280602Z",
     "iopub.status.idle": "2026-09-06T12:32:11.771019Z",
     "shell.execute_reply.started": "2026-09-06T12:32:08.280569Z",
     "shell.execute_reply": "2026-09-06T12:32:11.769922Z"
    }
   },
   "outputs": [],
   "execution_count": 12
  },
  {
   "cell_type": "code",
   "source": "import os\nimport sys\nimport shutil\nimport glob\nimport json\nimport re\n\n###################################################################\n# CONFIGURATION\n###################################################################\nDATASET_ROOT = '/kaggle/input/datasets/gokulrocky/polypdb-polyp-raw/PolypDB/PolypDB/PolypDB_center_wise/BKAI'\nVERIFY_DATASET_NAME = \"PolypDB_BKAI\"\nWORKING_DATASET = '/kaggle/working/dataset_copy'\n\n# 1. Clean up old working directory\nif os.path.exists(WORKING_DATASET):\n    shutil.rmtree(WORKING_DATASET)\n\n# 2. Create the flat structure that the Harness strictly requires\nos.makedirs(os.path.join(WORKING_DATASET, 'images'))\nos.makedirs(os.path.join(WORKING_DATASET, 'masks'))\n\nprint(f\"Flattening and copying files from all modalities in {DATASET_ROOT} ...\")\nimage_count = 0\nmask_count = 0\n\n# 3. Recursively hunt down every image and mask in FICE, BLI, WLI, LCI\nfor root, dirs, files in os.walk(DATASET_ROOT):\n    for file in files:\n        # Ignore random hidden files\n        if file.startswith('.') or not file.endswith(('.png', '.jpg', '.tif', '.bmp')):\n            continue\n            \n        src = os.path.join(root, file)\n        \n        # We prefix the filename with the modality name (e.g. FICE_image1.png) \n        # so files with the same name from different modalities don't overwrite each other!\n        modality = os.path.basename(os.path.dirname(root)) \n        new_filename = f\"{modality}_{file}\"\n        \n        if 'images' in root.lower():\n            dst = os.path.join(WORKING_DATASET, 'images', new_filename)\n            shutil.copy2(src, dst)\n            image_count += 1\n        elif 'masks' in root.lower():\n            dst = os.path.join(WORKING_DATASET, 'masks', new_filename)\n            shutil.copy2(src, dst)\n            mask_count += 1\n\nprint(f\"\u2705 Flattening Complete!\")\nprint(f\"Copied {image_count} real images and {mask_count} real masks into {WORKING_DATASET}\")",
   "metadata": {
    "trusted": true,
    "execution": {
     "iopub.status.busy": "2026-09-06T12:32:11.773055Z",
     "iopub.execute_input": "2026-09-06T12:32:11.773442Z",
     "iopub.status.idle": "2026-09-06T12:32:14.688316Z",
     "shell.execute_reply.started": "2026-09-06T12:32:11.773409Z",
     "shell.execute_reply": "2026-09-06T12:32:14.687670Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "text": "Flattening and copying files from all modalities in /kaggle/input/datasets/gokulrocky/polypdb-polyp-raw/PolypDB/PolypDB/PolypDB_center_wise/BKAI ...\n\u2705 Flattening Complete!\nCopied 1200 real images and 1200 real masks into /kaggle/working/dataset_copy\n",
     "output_type": "stream"
    }
   ],
   "execution_count": 13
  },
  {
   "cell_type": "code",
   "source": "import os\n\ndef print_directory_tree(startpath, max_depth=4):\n    print(f\"\ud83d\uddfa\ufe0f Directory tree for: {startpath}\")\n    print(\"=\" * 50)\n    \n    for root, dirs, files in os.walk(startpath):\n        level = root.replace(startpath, '').count(os.sep)\n        \n        # Stop digging if we go too deep (keeps the output readable)\n        if level > max_depth:\n            continue\n            \n        indent = ' ' * 4 * level\n        folder_name = os.path.basename(root)\n        if folder_name == '':\n            folder_name = startpath\n            \n        print(f\"{indent}\ud83d\udcc2 {folder_name}/\")\n        \n        subindent = ' ' * 4 * (level + 1)\n        \n        # Print the first 3 files in this folder so we can see what type of data is here\n        for f in files[:3]:\n            print(f\"{subindent}\ud83d\udcc4 {f}\")\n            \n        if len(files) > 3:\n            print(f\"{subindent}... and {len(files)-3} more files\")\n\nprint_directory_tree('/kaggle/input', max_depth=4)",
   "metadata": {
    "trusted": true,
    "execution": {
     "iopub.status.busy": "2026-09-06T12:32:14.689178Z",
     "iopub.execute_input": "2026-09-06T12:32:14.689378Z",
     "iopub.status.idle": "2026-09-06T12:32:19.280929Z",
     "shell.execute_reply.started": "2026-09-06T12:32:14.689356Z",
     "shell.execute_reply": "2026-09-06T12:32:19.280303Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "text": "\ud83d\uddfa\ufe0f Directory tree for: /kaggle/input\n==================================================\n\ud83d\udcc2 input/\n    \ud83d\udcc2 datasets/\n        \ud83d\udcc2 gokulraj324/\n            \ud83d\udcc2 anti-fabrication-toolkit/\n                \ud83d\udcc4 harness_core.py\n        \ud83d\udcc2 gokulrocky/\n            \ud83d\udcc2 chakramodel-kaggle-code/\n                \ud83d\udcc4 requirements.txt\n                \ud83d\udcc2 src/\n                    \ud83d\udcc4 hybrid_refine.py\n                    \ud83d\udcc4 train_yolo.py\n                    \ud83d\udcc4 train_pranet.py\n                    ... and 31 more files\n            \ud83d\udcc2 polypdb-polyp-raw/\n                \ud83d\udcc2 PolypDB/\n            \ud83d\udcc2 final-om-evlautation-upload/\n                \ud83d\udcc4 Kaggle_ChakraTransformer_Evaluation.ipynb\n                \ud83d\udcc2 weights/\n                    \ud83d\udcc4 chakra_transformer_best.pth\n                    \ud83d\udcc4 best.pt\n                \ud83d\udcc2 src/\n                    \ud83d\udcc4 hybrid_refine.py\n                    \ud83d\udcc4 train_yolo.py\n                    \ud83d\udcc4 metrics_engine_v2.py\n                    ... and 32 more files\n",
     "output_type": "stream"
    }
   ],
   "execution_count": 14
  },
  {
   "cell_type": "code",
   "source": "%%writefile /kaggle/working/verify_kaggle.py\nimport os\nimport sys\nimport glob\nimport cv2\nimport numpy as np\nimport torch\nfrom pathlib import Path\nimport hashlib\n\ndef md5(fname):\n    hash_md5 = hashlib.md5()\n    with open(fname, \"rb\") as f:\n        for chunk in iter(lambda: f.read(4096), b\"\"):\n            hash_md5.update(chunk)\n    return hash_md5.hexdigest()\n\ndef verify_strict():\n    print(\"=\" * 60)\n    print(\"STRICT CHECKPOINT AND ARCHITECTURE VERIFICATION\")\n    print(\"=\" * 60)\n    \n    # Identify codebase path\n    code_dir = None\n    for root, dirs, files in os.walk('/kaggle/input'):\n        if 'chakranet_segmenter.py' in files and 'src' in root:\n            code_dir = root\n            break\n    if not code_dir:\n        for root, dirs, files in os.walk('/kaggle/input'):\n            if 'chakranet_segmenter.py' in files:\n                code_dir = root\n                break\n    assert code_dir, \"FATAL: ChakraModel codebase not found in /kaggle/input\"\n    sys.path.insert(0, code_dir)\n    print(f\"[VERIFY] Codebase found at: {code_dir}\")\n    \n    from ultralytics import YOLO\n    from chakranet_segmenter import ChakraNet\n    from metrics_engine_v2 import MetricsEngineV2\n\n    # BUG 3 FIX: Strict Weight Selection\n    yolo_candidates = glob.glob('/kaggle/input/**/best.pt', recursive=True)\n    vit_candidates = glob.glob('/kaggle/input/**/chakra_transformer_best.pth', recursive=True)\n    \n    # Strictly filter yolo_candidates (ignore anything with 'transformer' in the path)\n    yolo_candidates = [p for p in yolo_candidates if 'transformer' not in p.lower()]\n    \n    print(f\"YOLO candidates found: {yolo_candidates}\")\n    print(f\"ViT candidates found: {vit_candidates}\")\n    \n    assert len(yolo_candidates) == 1, f\"FATAL: Expected exactly 1 YOLO weight best.pt, found {len(yolo_candidates)}!\"\n    assert len(vit_candidates) == 1, f\"FATAL: Expected exactly 1 ViT weight chakra_transformer_best.pth, found {len(vit_candidates)}!\"\n    \n    yolo_path = yolo_candidates[0]\n    weight_path = vit_candidates[0]\n\n    print(f\"Segmenter Checkpoint: {weight_path}\")\n    print(f\"Segmenter MD5: {md5(weight_path)}\")\n    print(f\"YOLO Checkpoint: {yolo_path}\")\n    print(f\"YOLO MD5: {md5(yolo_path)}\")\n    print(\"=\" * 60)\n    \n    nonce = os.environ.get(\"VERIFY_NONCE\", \"NO_NONCE\")\n    print(f\"VERIFY_NONCE: {nonce}\")\n    \n    device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    print(f\"Executing on device: {device}\")\n    \n    yolo_model = YOLO(yolo_path)\n    yolo_model.to(device)\n    \n    segmenter = ChakraNet(img_size=(384, 384), device=device)\n    segmenter.model.to(device)\n    \n    # Load weights correctly by explicitly stripping 'module.'\n    sd = torch.load(weight_path, map_location=device)\n    sd_fixed = {k.replace(\"_orig_mod.\", \"\").replace(\"module.\", \"\"): v for k, v in sd.items()}\n    res = segmenter.model.load_state_dict(sd_fixed, strict=False)\n    print(f\"ChakraNet Weights Loaded. Missing keys: {len(res.missing_keys)}, Unexpected keys: {len(res.unexpected_keys)}\")\n    \n    total_keys = len(segmenter.model.state_dict())\n    if len(res.missing_keys) > (0.1 * total_keys):\n        print(f\"[FATAL] Major weight mismatch! Missing keys > 10% ({len(res.missing_keys)} missing out of {total_keys}).\")\n        sys.exit(1)\n    \n    metrics_engine = MetricsEngineV2()\n    \n    dataset_name = os.environ.get(\"VERIFY_DATASET_NAME\")\n    dataset_root = os.environ.get(\"VERIFY_DATASET_ROOT\")\n    \n    if not dataset_name or not dataset_root:\n        print(\"[FATAL] VERIFY_DATASET_NAME or VERIFY_DATASET_ROOT environment variable not set.\")\n        sys.exit(1)\n        \n    print(f\"\\n============================================================\")\n    print(f\"VERIFYING DATASET: {dataset_name}\")\n    print(f\"============================================================\")\n    \n    d_root = Path(dataset_root)\n    images_dir = d_root / \"images\"\n    masks_dir = d_root / \"masks\"\n    \n    if not images_dir.exists():\n        print(f\"[FATAL] Images directory not found: {images_dir}\")\n        sys.exit(1)\n        \n    image_paths = sorted(list(images_dir.glob(\"*.png\")) + list(images_dir.glob(\"*.jpg\")) + list(images_dir.glob(\"*.tif\")))\n    print(f\"Total images found: {len(image_paths)}\")\n    \n    results = {k: [] for k in [\"Dice\", \"mIoU\", \"wF-measure\", \"S-measure\", \"E-measure\", \"MAE\"]}\n    import csv\n    csv_file = \"/kaggle/working/per_image_metrics_log.csv\"\n    with open(csv_file, 'w', newline='') as f:\n        writer = csv.writer(f)\n        writer.writerow([\"Filename\", \"Dice\", \"mIoU\", \"wF-measure\", \"S-measure\", \"E-measure\", \"MAE\"])\n        \n    for i, img_path in enumerate(image_paths):\n        mask_path = None\n        for ext in [\".png\", \".jpg\", \".tif\", \".bmp\"]:\n            potential_path = masks_dir / (img_path.stem + ext)\n            if potential_path.exists():\n                mask_path = potential_path\n                break\n                \n        if not mask_path:\n            print(f\"[WARN] Mask not found for {img_path.name}\")\n            continue\n            \n        img_bgr = cv2.imread(str(img_path))\n        gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)\n        gt_resized = cv2.resize(gt_mask, (448, 448), interpolation=cv2.INTER_NEAREST)\n        \n        img_resized = cv2.resize(img_bgr, (448, 448))\n        yolo_results = yolo_model(img_resized, verbose=False)[0]\n        \n        # Soft prob map initialized to 0\n        c_prob = np.zeros((448, 448), dtype=np.float32)\n        \n        if len(yolo_results.boxes) > 0:\n            boxes = sorted(yolo_results.boxes, key=lambda b: b.conf[0].item(), reverse=True)\n            box = boxes[0].xyxy[0].cpu().numpy()\n            x1, y1, x2, y2 = map(int, box)\n            \n            w_box, h_box = x2 - x1, y2 - y1\n            px, py = int(0.25 * w_box), int(0.25 * h_box)\n            px1, py1 = max(0, x1 - px), max(0, y1 - py)\n            px2, py2 = max(0, min(448, x2 + px)), max(0, min(448, y2 + py))\n            \n            if px2 > px1 and py2 > py1:\n                roi_crop = img_resized[py1:py2, px1:px2]\n                \n                # BUG 2 FIX: Direct manual inference EXACTLY matching chakranet_segmenter.py\n                from utils.transforms import letterbox_pad, unletterbox\n                img_rgb = cv2.cvtColor(roi_crop, cv2.COLOR_BGR2RGB)\n                img_resized_crop, meta = letterbox_pad(img_rgb, target_size=(384, 384))\n                \n                img_norm = img_resized_crop.astype(np.float32) / 255.0\n                mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)\n                std = np.array([0.229, 0.224, 0.225], dtype=np.float32)\n                img_norm = (img_norm - mean) / std\n                img_tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).to(device)\n                \n                try:\n                    with torch.no_grad():\n                        logits = segmenter.model(img_tensor)\n                        prob = torch.sigmoid(logits)\n                except RuntimeError as e:\n                    if \"out of memory\" in str(e).lower():\n                        print(\"[WARN] Switched to CPU due to CUDA OutOfMemory!\")\n                        img_tensor_cpu = img_tensor.cpu()\n                        segmenter.model.to('cpu')\n                        with torch.no_grad():\n                            logits = segmenter.model(img_tensor_cpu)\n                            prob = torch.sigmoid(logits)\n                        segmenter.model.to(device)\n                    else:\n                        raise\n                        \n                prob_np = prob.squeeze().cpu().numpy()\n                prob_resized = unletterbox(prob_np, meta)\n                \n                c_prob[py1:py2, px1:px2] = prob_resized\n        \n        # Explicit binarization ONLY for the ground truth here (network outputs soft prob)\n        gt_bin = (gt_resized > 127).astype(np.uint8)\n        \n        # Pass continuous probability map (c_prob) directly to compute_all \n        # (compute_all handles thresholding for Dice/IoU internally)\n        metrics = metrics_engine.compute_all(c_prob, gt_bin)\n        \n        for k in results.keys():\n            results[k].append(metrics[k])\n            \n        # BUG 4 FIX: Checkpoint saving per image\n        with open(csv_file, 'a', newline='') as f:\n            writer = csv.writer(f)\n            writer.writerow([img_path.name, metrics['Dice'], metrics['mIoU'], metrics['wF-measure'], metrics['S-measure'], metrics['E-measure'], metrics['MAE']])\n            \n        # IMPORTANT: Print per-image dice WITH FILENAME for canary detection\n        dice_score = metrics['Dice']\n        print(f\"File: {img_path.name} | Dice: {dice_score:.4f}\")\n        \n    print(f\"\\n[DONE] {dataset_name}\")\n    print(f\"Total Evaluated Images: {len(results['Dice'])}\")\n    for k in results.keys():\n        print(f\"Final True Average {k}: {float(np.mean(results[k])):.4f}\")\n    print(f\"============================================================\")\n\nif __name__ == \"__main__\":\n    verify_strict()\n\n",
   "metadata": {
    "trusted": true,
    "execution": {
     "iopub.status.busy": "2026-09-06T12:32:19.282464Z",
     "iopub.execute_input": "2026-09-06T12:32:19.282747Z",
     "iopub.status.idle": "2026-09-06T12:32:19.291453Z",
     "shell.execute_reply.started": "2026-09-06T12:32:19.282721Z",
     "shell.execute_reply": "2026-09-06T12:32:19.290842Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "text": "Overwriting /kaggle/working/verify_kaggle.py\n",
     "output_type": "stream"
    }
   ],
   "execution_count": 15
  },
  {
   "cell_type": "code",
   "source": "harness_path = glob.glob('/kaggle/input/**/harness_core.py', recursive=True)\nif not harness_path:\n    harness_path = glob.glob('/kaggle/working/**/harness_core.py', recursive=True)\n\nif not harness_path:\n    print(\"FATAL: harness_core.py not found! Did you attach the Anti-Fabrication Toolkit dataset?\")\n    sys.exit(1)\n\ntoolkit_dir = os.path.dirname(harness_path[0])\nsys.path.insert(0, toolkit_dir)\n\nfrom harness_core import plant_multi_canary, run_live, verify, sign_verdict\n\nprint(\"Planting canaries...\")\nsecret = plant_multi_canary(WORKING_DATASET)\n\nextra_env = {\n    \"VERIFY_NONCE\": secret[\"nonce\"],\n    \"VERIFY_DATASET_NAME\": VERIFY_DATASET_NAME,\n    \"VERIFY_DATASET_ROOT\": WORKING_DATASET\n}\n\nprint(f\"\\nRunning verification for {VERIFY_DATASET_NAME}...\")\nstdout_text, arrivals, elapsed, retcode = run_live('/kaggle/working/verify_kaggle.py', extra_env, timeout_sec=10000)\n\nprint(stdout_text)\n\nif retcode != 0:\n    print(f\"FATAL: verify_kaggle.py exited with code {retcode}\")\nelse:\n    # Parse claimed total from stdout\n    match = re.search(r\"Total Evaluated Images:\\s*(\\d+)\", stdout_text)\n    claimed_total = int(match.group(1)) if match else -1\n    \n    # Use harness core to verify output logic\n    real_count = len(os.listdir(os.path.join(WORKING_DATASET, \"images\")))\n    verdict_dict = verify(stdout_text, arrivals, elapsed, secret, claimed_total, real_count)\n    \n    card = {\n        \"project\": \"ChakraModel\",\n        \"dataset\": VERIFY_DATASET_NAME,\n        \"nonce\": secret[\"nonce\"],\n        \"decision\": \"ADMITTED\" if verdict_dict[\"pass\"] else \"WITHHELD\",\n        \"findings\": verdict_dict[\"findings\"],\n        \"elapsed_sec\": verdict_dict[\"elapsed_sec\"]\n    }\n    \n    signed_card = sign_verdict(card, secret[\"hmac_key\"])\n    \n    with open('/kaggle/working/completion_card.json', 'w') as f:\n        json.dump(signed_card, f, indent=2)\n        \n    print(f\"\\n=========================================\")\n    print(f\"VERDICT: {card['decision']}\")\n    for finding in card['findings']:\n        print(f\"  - {finding}\")\n    print(f\"Saved to completion_card.json\")\n    print(f\"=========================================\")\n",
   "metadata": {
    "trusted": true,
    "execution": {
     "iopub.status.busy": "2026-09-06T12:32:19.292461Z",
     "iopub.execute_input": "2026-09-06T12:32:19.292792Z",
     "iopub.status.idle": "2026-09-06T12:32:25.049920Z",
     "shell.execute_reply.started": "2026-09-06T12:32:19.292743Z",
     "shell.execute_reply": "2026-09-06T12:32:25.048915Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "text": "Planting canaries...\n\nRunning verification for PolypDB_BKAI...\n============================================================\nSTRICT CHECKPOINT AND ARCHITECTURE VERIFICATION\n============================================================\n[VERIFY] Codebase found at: /kaggle/input/datasets/gokulrocky/chakramodel-kaggle-code/src\n[WARN] HardwareMonitor unavailable: [Errno 30] Read-only file system: '/kaggle/input/datasets/gokulrocky/chakramodel-kaggle-code/logs'\nTraceback (most recent call last):\n  File \"/kaggle/working/verify_kaggle.py\", line 210, in <module>\n    verify_strict()\n  File \"/kaggle/working/verify_kaggle.py\", line 39, in verify_strict\n    from metrics_engine_v2 import MetricsEngineV2\nModuleNotFoundError: No module named 'metrics_engine_v2'\n\nFATAL: verify_kaggle.py exited with code 1\n",
     "output_type": "stream"
    }
   ],
   "execution_count": 16
  },
  {
   "cell_type": "code",
   "source": "import os\nbkai_path = '/kaggle/input/datasets/gokulrocky/polypdb-polyp-raw/PolypDB/PolypDB/PolypDB_center_wise/BKAI'\n\nprint(f\"Contents of BKAI:\")\nitems = os.listdir(bkai_path)\nfor item in items[:20]:\n    path = os.path.join(bkai_path, item)\n    if os.path.isdir(path):\n        print(f\"\ud83d\udcc2 [FOLDER] {item}\")\n    else:\n        print(f\"\ud83d\udcc4 [FILE]   {item}\")\n        \nif len(items) > 20:\n    print(f\"... and {len(items)-20} more items.\")",
   "metadata": {
    "trusted": true,
    "execution": {
     "iopub.status.busy": "2026-09-06T12:32:25.051020Z",
     "iopub.execute_input": "2026-09-06T12:32:25.051365Z",
     "iopub.status.idle": "2026-09-06T12:32:25.058136Z",
     "shell.execute_reply.started": "2026-09-06T12:32:25.051331Z",
     "shell.execute_reply": "2026-09-06T12:32:25.057313Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "text": "Contents of BKAI:\n\ud83d\udcc2 [FOLDER] FICE\n\ud83d\udcc2 [FOLDER] BLI\n\ud83d\udcc2 [FOLDER] WLI\n\ud83d\udcc2 [FOLDER] LCI\n",
     "output_type": "stream"
    }
   ],
   "execution_count": 17
  },
  {
   "cell_type": "code",
   "source": "!pip install ultralytics",
   "metadata": {
    "trusted": true,
    "execution": {
     "iopub.status.busy": "2026-09-06T12:32:25.058928Z",
     "iopub.execute_input": "2026-09-06T12:32:25.059116Z",
     "iopub.status.idle": "2026-09-06T12:32:28.793973Z",
     "shell.execute_reply.started": "2026-09-06T12:32:25.059095Z",
     "shell.execute_reply": "2026-09-06T12:32:28.792839Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "text": "Requirement already satisfied: ultralytics in /usr/local/lib/python3.12/dist-packages (8.4.142)\nRequirement already satisfied: cloudpickle>=3.1.1 in /usr/local/lib/python3.12/dist-packages (from ultralytics) (3.1.2)\nRequirement already satisfied: filelock>=3.16.1 in /usr/local/lib/python3.12/dist-packages (from ultralytics) (3.29.0)\nRequirement already satisfied: numpy>=1.23.0 in /usr/local/lib/python3.12/dist-packages (from ultralytics) (2.0.2)\nRequirement already satisfied: matplotlib>=3.3.0 in /usr/local/lib/python3.12/dist-packages (from ultralytics) (3.10.0)\nRequirement already satisfied: opencv-python!=4.13.0.90,>=4.7.0 in /usr/local/lib/python3.12/dist-packages (from ultralytics) (4.13.0.92)\nRequirement already satisfied: pillow>=7.1.2 in /usr/local/lib/python3.12/dist-packages (from ultralytics) (11.3.0)\nRequirement already satisfied: pyyaml>=5.3.1 in /usr/local/lib/python3.12/dist-packages (from ultralytics) (6.0.3)\nRequirement already satisfied: requests>=2.23.0 in /usr/local/lib/python3.12/dist-packages (from ultralytics) (2.32.4)\nRequirement already satisfied: torch>=1.8.0 in /usr/local/lib/python3.12/dist-packages (from ultralytics) (2.10.0+cu128)\nRequirement already satisfied: torchvision>=0.9.0 in /usr/local/lib/python3.12/dist-packages (from ultralytics) (0.25.0+cu128)\nRequirement already satisfied: psutil>=5.8.0 in /usr/local/lib/python3.12/dist-packages (from ultralytics) (5.9.5)\nRequirement already satisfied: polars>=0.20.0 in /usr/local/lib/python3.12/dist-packages (from ultralytics) (1.35.2)\nRequirement already satisfied: nvidia-ml-py>=12.0.0 in /usr/local/lib/python3.12/dist-packages (from ultralytics) (13.595.45)\nRequirement already satisfied: ultralytics-thop>=2.1.6 in /usr/local/lib/python3.12/dist-packages (from ultralytics) (2.1.6)\nRequirement already satisfied: ultralytics-platform>=0.1.3 in /usr/local/lib/python3.12/dist-packages (from ultralytics) (0.1.21)\nRequirement already satisfied: contourpy>=1.0.1 in /usr/local/lib/python3.12/dist-packages (from matplotlib>=3.3.0->ultralytics) (1.3.3)\nRequirement already satisfied: cycler>=0.10 in /usr/local/lib/python3.12/dist-packages (from matplotlib>=3.3.0->ultralytics) (0.12.1)\nRequirement already satisfied: fonttools>=4.22.0 in /usr/local/lib/python3.12/dist-packages (from matplotlib>=3.3.0->ultralytics) (4.62.1)\nRequirement already satisfied: kiwisolver>=1.3.1 in /usr/local/lib/python3.12/dist-packages (from matplotlib>=3.3.0->ultralytics) (1.5.0)\nRequirement already satisfied: packaging>=20.0 in /usr/local/lib/python3.12/dist-packages (from matplotlib>=3.3.0->ultralytics) (26.1)\nRequirement already satisfied: pyparsing>=2.3.1 in /usr/local/lib/python3.12/dist-packages (from matplotlib>=3.3.0->ultralytics) (3.3.2)\nRequirement already satisfied: python-dateutil>=2.7 in /usr/local/lib/python3.12/dist-packages (from matplotlib>=3.3.0->ultralytics) (2.9.0.post0)\nRequirement already satisfied: polars-runtime-32==1.35.2 in /usr/local/lib/python3.12/dist-packages (from polars>=0.20.0->ultralytics) (1.35.2)\nRequirement already satisfied: charset_normalizer<4,>=2 in /usr/local/lib/python3.12/dist-packages (from requests>=2.23.0->ultralytics) (3.4.7)\nRequirement already satisfied: idna<4,>=2.5 in /usr/local/lib/python3.12/dist-packages (from requests>=2.23.0->ultralytics) (3.13)\nRequirement already satisfied: urllib3<3,>=1.21.1 in /usr/local/lib/python3.12/dist-packages (from requests>=2.23.0->ultralytics) (2.5.0)\nRequirement already satisfied: certifi>=2017.4.17 in /usr/local/lib/python3.12/dist-packages (from requests>=2.23.0->ultralytics) (2026.4.22)\nRequirement already satisfied: typing-extensions>=4.10.0 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (4.15.0)\nRequirement already satisfied: setuptools in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (81.0.0)\nRequirement already satisfied: sympy>=1.13.3 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (1.14.0)\nRequirement already satisfied: networkx>=2.5.1 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (3.6.1)\nRequirement already satisfied: jinja2 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (3.1.6)\nRequirement already satisfied: fsspec>=0.8.5 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (2025.3.0)\nRequirement already satisfied: cuda-bindings==12.9.4 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (12.9.4)\nRequirement already satisfied: nvidia-cuda-nvrtc-cu12==12.8.93 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (12.8.93)\nRequirement already satisfied: nvidia-cuda-runtime-cu12==12.8.90 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (12.8.90)\nRequirement already satisfied: nvidia-cuda-cupti-cu12==12.8.90 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (12.8.90)\nRequirement already satisfied: nvidia-cudnn-cu12==9.10.2.21 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (9.10.2.21)\nRequirement already satisfied: nvidia-cublas-cu12==12.8.4.1 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (12.8.4.1)\nRequirement already satisfied: nvidia-cufft-cu12==11.3.3.83 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (11.3.3.83)\nRequirement already satisfied: nvidia-curand-cu12==10.3.9.90 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (10.3.9.90)\nRequirement already satisfied: nvidia-cusolver-cu12==11.7.3.90 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (11.7.3.90)\nRequirement already satisfied: nvidia-cusparse-cu12==12.5.8.93 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (12.5.8.93)\nRequirement already satisfied: nvidia-cusparselt-cu12==0.7.1 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (0.7.1)\nRequirement already satisfied: nvidia-nccl-cu12==2.27.5 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (2.27.5)\nRequirement already satisfied: nvidia-nvshmem-cu12==3.4.5 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (3.4.5)\nRequirement already satisfied: nvidia-nvtx-cu12==12.8.90 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (12.8.90)\nRequirement already satisfied: nvidia-nvjitlink-cu12==12.8.93 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (12.8.93)\nRequirement already satisfied: nvidia-cufile-cu12==1.13.1.3 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (1.13.1.3)\nRequirement already satisfied: triton==3.6.0 in /usr/local/lib/python3.12/dist-packages (from torch>=1.8.0->ultralytics) (3.6.0)\nRequirement already satisfied: cuda-pathfinder~=1.1 in /usr/local/lib/python3.12/dist-packages (from cuda-bindings==12.9.4->torch>=1.8.0->ultralytics) (1.5.3)\nRequirement already satisfied: httpx<1,>=0.28 in /usr/local/lib/python3.12/dist-packages (from ultralytics-platform>=0.1.3->ultralytics) (0.28.1)\nRequirement already satisfied: anyio in /usr/local/lib/python3.12/dist-packages (from httpx<1,>=0.28->ultralytics-platform>=0.1.3->ultralytics) (4.13.0)\nRequirement already satisfied: httpcore==1.* in /usr/local/lib/python3.12/dist-packages (from httpx<1,>=0.28->ultralytics-platform>=0.1.3->ultralytics) (1.0.9)\nRequirement already satisfied: h11>=0.16 in /usr/local/lib/python3.12/dist-packages (from httpcore==1.*->httpx<1,>=0.28->ultralytics-platform>=0.1.3->ultralytics) (0.16.0)\nRequirement already satisfied: six>=1.5 in /usr/local/lib/python3.12/dist-packages (from python-dateutil>=2.7->matplotlib>=3.3.0->ultralytics) (1.17.0)\nRequirement already satisfied: mpmath<1.4,>=1.1.0 in /usr/local/lib/python3.12/dist-packages (from sympy>=1.13.3->torch>=1.8.0->ultralytics) (1.3.0)\nRequirement already satisfied: MarkupSafe>=2.0 in /usr/local/lib/python3.12/dist-packages (from jinja2->torch>=1.8.0->ultralytics) (3.0.3)\n",
     "output_type": "stream"
    }
   ],
   "execution_count": 18
  },
  {
   "cell_type": "code",
   "source": [
    "import os\n",
    "import sys\n",
    "import glob\n",
    "import cv2\n",
    "import numpy as np\n",
    "import torch\n",
    "from pathlib import Path\n",
    "\n",
    "# ---------------------------------------------------------------------------\n",
    "# Locate ChakraModel codebase\n",
    "# ---------------------------------------------------------------------------\n",
    "# Add all src directories in Kaggle datasets to sys.path to ensure we find everything\n",
    "src_dirs_added = []\n",
    "for root, dirs, files in os.walk('/kaggle/input'):\n",
    "    if os.path.basename(root) == 'src':\n",
    "        sys.path.insert(0, root)\n",
    "        src_dirs_added.append(root)\n",
    "\n",
    "if not src_dirs_added:\n",
    "    # Fallback just in case\n",
    "    for root, dirs, files in os.walk('/kaggle/input'):\n",
    "        if 'chakranet_segmenter.py' in files:\n",
    "            sys.path.insert(0, root)\n",
    "            src_dirs_added.append(root)\n",
    "\n",
    "print(f\"[REPRODUCIBILITY] Added {len(src_dirs_added)} codebase directories to sys.path:\")\n",
    "for d in src_dirs_added:\n",
    "    print(f\"  -> {d}\")\n",
    "\n",
    "from ultralytics import YOLO\n",
    "from chakranet_segmenter import ChakraNet\n",
    "from metrics_engine_v2 import MetricsEngineV2\n",
    "\n",
    "\n",
    "def compute_iou(box1, box2):\n",
    "    x_left = max(box1[0], box2[0])\n",
    "    y_top = max(box1[1], box2[1])\n",
    "    x_right = min(box1[2], box2[2])\n",
    "    y_bottom = min(box1[3], box2[3])\n",
    "\n",
    "    if x_right < x_left or y_bottom < y_top:\n",
    "        return 0.0\n",
    "\n",
    "    intersection_area = (x_right - x_left) * (y_bottom - y_top)\n",
    "    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])\n",
    "    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])\n",
    "    iou = intersection_area / float(box1_area + box2_area - intersection_area + 1e-6)\n",
    "    return iou\n",
    "\n",
    "\n",
    "def get_gt_bbox(mask_img):\n",
    "    coords = cv2.findNonZero(mask_img)\n",
    "    if coords is None:\n",
    "        return None\n",
    "    x, y, w, h = cv2.boundingRect(coords)\n",
    "    return [x, y, x + w, y + h]\n",
    "\n",
    "\n",
    "def run_diagnostics():\n",
    "    print(\"=\" * 60)\n",
    "    print(\"STARTING DIAGNOSTIC EVALUATION (YOLO vs ChakraNet)\")\n",
    "    print(\"=\" * 60)\n",
    "\n",
    "    # -----------------------------------------------------------------------\n",
    "    # Locate weights - FIX #4: print exactly what's chosen, don't pick silently\n",
    "    # -----------------------------------------------------------------------\n",
    "    yolo_candidates = sorted(glob.glob('/kaggle/input/**/best.pt', recursive=True))\n",
    "    vit_candidates = sorted(glob.glob('/kaggle/input/**/chakra_transformer_best.pth', recursive=True))\n",
    "\n",
    "    if not yolo_candidates or not vit_candidates:\n",
    "        print(\"FATAL: could not find YOLO best.pt or chakra_transformer_best.pth\")\n",
    "        sys.exit(1)\n",
    "\n",
    "    if len(yolo_candidates) > 1:\n",
    "        print(f\"[WARNING] Multiple YOLO checkpoints found: {yolo_candidates}\")\n",
    "    if len(vit_candidates) > 1:\n",
    "        print(f\"[WARNING] Multiple ChakraNet checkpoints found: {vit_candidates}\")\n",
    "\n",
    "    yolo_path = yolo_candidates[-1]\n",
    "    weight_path = vit_candidates[-1]\n",
    "    print(f\"[REPRODUCIBILITY] YOLO checkpoint : {yolo_path}\")\n",
    "    print(f\"[REPRODUCIBILITY] ChakraNet ckpt  : {weight_path}\")\n",
    "\n",
    "    device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n",
    "    print(f\"[INFO] Device: {device}\")\n",
    "\n",
    "    # Load YOLO\n",
    "    yolo_model = YOLO(yolo_path)\n",
    "    yolo_model.to(device)\n",
    "\n",
    "    # -----------------------------------------------------------------------\n",
    "    # Load ChakraNet - FIX #2: strict loading with explicit key reporting\n",
    "    # -----------------------------------------------------------------------\n",
    "    segmenter = ChakraNet(img_size=(384, 384), device=device)\n",
    "    segmenter.model.to(device)\n",
    "    sd = torch.load(weight_path, map_location=device)\n",
    "    sd_fixed = {k.replace(\"_orig_mod.\", \"\").replace(\"module.\", \"\"): v for k, v in sd.items()}\n",
    "\n",
    "    load_result = segmenter.model.load_state_dict(sd_fixed, strict=False)\n",
    "    if load_result.missing_keys:\n",
    "        print(f\"[CHECKPOINT WARNING] {len(load_result.missing_keys)} missing keys:\")\n",
    "        for k in load_result.missing_keys[:20]:\n",
    "            print(f\"    MISSING: {k}\")\n",
    "        if len(load_result.missing_keys) > 20:\n",
    "            print(f\"    ... and {len(load_result.missing_keys) - 20} more\")\n",
    "    if load_result.unexpected_keys:\n",
    "        print(f\"[CHECKPOINT WARNING] {len(load_result.unexpected_keys)} unexpected keys:\")\n",
    "        for k in load_result.unexpected_keys[:20]:\n",
    "            print(f\"    UNEXPECTED: {k}\")\n",
    "        if len(load_result.unexpected_keys) > 20:\n",
    "            print(f\"    ... and {len(load_result.unexpected_keys) - 20} more\")\n",
    "    if not load_result.missing_keys and not load_result.unexpected_keys:\n",
    "        print(\"[CHECKPOINT OK] All keys matched exactly.\")\n",
    "    else:\n",
    "        print(\"[CHECKPOINT WARNING] Mismatches above mean results may not reflect \"\n",
    "              \"your intended trained weights. Investigate before trusting metrics below.\")\n",
    "\n",
    "    # FIX #1: eval mode - THIS WAS MISSING IN THE ORIGINAL DRAFT\n",
    "    segmenter.model.eval()\n",
    "    print(\"[FIX APPLIED] segmenter.model.eval() called - BN/Dropout now in inference mode.\")\n",
    "\n",
    "    metrics_engine = MetricsEngineV2()\n",
    "\n",
    "    images_dir = Path(\"/kaggle/working/dataset_copy/images\")\n",
    "    masks_dir = Path(\"/kaggle/working/dataset_copy/masks\")\n",
    "\n",
    "    image_paths = sorted(list(images_dir.glob(\"*.png\")) + list(images_dir.glob(\"*.jpg\")) + list(images_dir.glob(\"*.tif\")))\n",
    "    image_paths = [p for p in image_paths if not p.name.startswith(\"CANARY\")]\n",
    "\n",
    "    print(f\"Total valid diagnostic images: {len(image_paths)}\")\n",
    "    print(\"[ASSUMPTION CHECK] Oracle crop uses 25% bbox padding, assumed to match \"\n",
    "          \"your E2E pipeline's YOLO->crop expansion logic. If your actual pipeline \"\n",
    "          \"uses a different padding ratio, the oracle number below is not apples-to-apples \"\n",
    "          \"with your full-pipeline Dice of 0.8370 - verify this before drawing conclusions.\")\n",
    "\n",
    "    yolo_ious = []\n",
    "    yolo_hits_any = 0       # IoU > 0.0\n",
    "    yolo_hits_50 = 0        # IoU >= 0.5  (FIX #3)\n",
    "    yolo_misses = 0\n",
    "\n",
    "    chakra_results = {k: [] for k in [\"Dice\", \"mIoU\", \"wF-measure\", \"S-measure\", \"E-measure\", \"MAE\"]}\n",
    "    per_image_dice = []     # FIX #5\n",
    "\n",
    "    for i, img_path in enumerate(image_paths):\n",
    "        mask_path = None\n",
    "        for ext in [\".png\", \".jpg\", \".tif\", \".bmp\"]:\n",
    "            potential = masks_dir / (img_path.stem + ext)\n",
    "            if potential.exists():\n",
    "                mask_path = potential\n",
    "                break\n",
    "\n",
    "        if not mask_path:\n",
    "            continue\n",
    "\n",
    "        img_bgr = cv2.imread(str(img_path))\n",
    "        gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)\n",
    "\n",
    "        img_resized = cv2.resize(img_bgr, (448, 448))\n",
    "        gt_resized = cv2.resize(gt_mask, (448, 448), interpolation=cv2.INTER_NEAREST)\n",
    "        gt_bin = (gt_resized > 127).astype(np.uint8)\n",
    "\n",
    "        gt_bbox = get_gt_bbox(gt_bin)\n",
    "\n",
    "        if gt_bbox is None:\n",
    "            continue\n",
    "\n",
    "        # ---------------------------------------------------------------\n",
    "        # 1. DIAGNOSTIC: YOLO ONLY (Detection)\n",
    "        # ---------------------------------------------------------------\n",
    "        yolo_results = yolo_model(img_resized, verbose=False)[0]\n",
    "        if len(yolo_results.boxes) > 0:\n",
    "            boxes = sorted(yolo_results.boxes, key=lambda b: b.conf[0].item(), reverse=True)\n",
    "            pred_box = boxes[0].xyxy[0].cpu().numpy().tolist()\n",
    "\n",
    "            iou = compute_iou(gt_bbox, pred_box)\n",
    "            yolo_ious.append(iou)\n",
    "            if iou > 0.0:\n",
    "                yolo_hits_any += 1\n",
    "            else:\n",
    "                yolo_misses += 1\n",
    "            if iou >= 0.5:\n",
    "                yolo_hits_50 += 1\n",
    "        else:\n",
    "            yolo_ious.append(0.0)\n",
    "            yolo_misses += 1\n",
    "\n",
    "        # ---------------------------------------------------------------\n",
    "        # 2. DIAGNOSTIC: CHAKRANET ONLY (Perfect Crops)\n",
    "        # ---------------------------------------------------------------\n",
    "        x1, y1, x2, y2 = gt_bbox\n",
    "        w_box, h_box = x2 - x1, y2 - y1\n",
    "\n",
    "        px, py = int(0.25 * w_box), int(0.25 * h_box)\n",
    "        px1, py1 = max(0, x1 - px), max(0, y1 - py)\n",
    "        px2, py2 = max(0, min(448, x2 + px)), max(0, min(448, y2 + py))\n",
    "\n",
    "        if px2 > px1 and py2 > py1:\n",
    "            roi_crop = img_resized[py1:py2, px1:px2]\n",
    "\n",
    "            img_rgb = cv2.cvtColor(roi_crop, cv2.COLOR_BGR2RGB)\n",
    "            img_resized_crop = cv2.resize(img_rgb, (384, 384))\n",
    "            img_norm = img_resized_crop.astype(np.float32) / 255.0\n",
    "            mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)\n",
    "            std = np.array([0.229, 0.224, 0.225], dtype=np.float32)\n",
    "            img_norm = (img_norm - mean) / std\n",
    "            img_tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).to(device)\n",
    "\n",
    "            with torch.no_grad():\n",
    "                logits = segmenter.model(img_tensor)\n",
    "                prob = torch.sigmoid(logits)\n",
    "\n",
    "            prob_np = prob.squeeze().float().cpu().numpy().astype(np.float32)\n",
    "            prob_resized = cv2.resize(prob_np, (px2 - px1, py2 - py1), interpolation=cv2.INTER_LINEAR)\n",
    "\n",
    "            c_prob = np.zeros((448, 448), dtype=np.float32)\n",
    "            c_prob[py1:py2, px1:px2] = prob_resized\n",
    "\n",
    "            metrics = metrics_engine.compute_all(c_prob, gt_bin)\n",
    "            for k in chakra_results.keys():\n",
    "                chakra_results[k].append(metrics[k])\n",
    "            per_image_dice.append(metrics[\"Dice\"])\n",
    "\n",
    "        if i % 100 == 0:\n",
    "            print(f\"Processed {i}/{len(image_paths)} images...\")\n",
    "\n",
    "    print(\"\\n=========================================\")\n",
    "    print(\"YOLO DETECTION PERFORMANCE (DIAGNOSTIC)\")\n",
    "    print(\"=========================================\")\n",
    "    print(f\"Average Bounding Box IoU: {np.mean(yolo_ious):.4f}\")\n",
    "    total_yolo = yolo_hits_any + yolo_misses\n",
    "    print(f\"Detection Hit Rate (IoU > 0.0):  {yolo_hits_any}/{total_yolo} ({yolo_hits_any/total_yolo*100:.2f}%)\")\n",
    "    print(f\"Detection Recall  (IoU >= 0.5):  {yolo_hits_50}/{total_yolo} ({yolo_hits_50/total_yolo*100:.2f}%)\")\n",
    "\n",
    "    print(\"\\n=========================================\")\n",
    "    print(\"CHAKRANET SEGMENTATION PERFORMANCE (PERFECT CROP)\")\n",
    "    print(\"=========================================\")\n",
    "    for k in chakra_results.keys():\n",
    "        print(f\"Perfect Crop Average {k}: {np.mean(chakra_results[k]):.4f}\")\n",
    "\n",
    "    if per_image_dice:\n",
    "        arr = np.array(per_image_dice)\n",
    "        print(\"\\n=========================================\")\n",
    "        print(\"PER-IMAGE DICE DISTRIBUTION (bimodal check)\")\n",
    "        print(\"=========================================\")\n",
    "        bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.01]\n",
    "        labels = [\"0.0-0.2\", \"0.2-0.4\", \"0.4-0.6\", \"0.6-0.8\", \"0.8-1.0\"]\n",
    "        counts, _ = np.histogram(arr, bins=bins)\n",
    "        for lbl, c in zip(labels, counts):\n",
    "            print(f\"  Dice {lbl}: {c} images ({c/len(arr)*100:.1f}%)\")\n",
    "        print(f\"  Median Dice: {np.median(arr):.4f}  |  Std Dice: {np.std(arr):.4f}\")\n",
    "        print(\"  -> If 0.0-0.2 bucket is large relative to a strong 0.8-1.0 peak, \"\n",
    "              \"that's a bimodal signal -> likely a DETECTION/localization problem \"\n",
    "              \"(some images fail hard), not smooth segmentation weakness.\")\n",
    "    print(\"=========================================\")\n",
    "\n",
    "\n",
    "if __name__ == \"__main__\":\n",
    "    run_diagnostics()\n"
   ],
   "metadata": {
    "trusted": true,
    "execution": {
     "iopub.status.busy": "2026-09-06T12:32:28.795751Z",
     "iopub.execute_input": "2026-09-06T12:32:28.796139Z",
     "iopub.status.idle": "2026-09-06T12:33:02.904979Z",
     "shell.execute_reply.started": "2026-09-06T12:32:28.796108Z",
     "shell.execute_reply": "2026-09-06T12:33:02.903990Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "text": "[REPRODUCIBILITY] Added 2 codebase directories to sys.path:\n  -> /kaggle/input/datasets/gokulrocky/chakramodel-kaggle-code/src\n  -> /kaggle/input/datasets/gokulrocky/final-om-evlautation-upload/src\n============================================================\nSTARTING DIAGNOSTIC EVALUATION (YOLO vs ChakraNet)\n============================================================\n[REPRODUCIBILITY] YOLO checkpoint : /kaggle/input/datasets/gokulrocky/final-om-evlautation-upload/weights/best.pt\n[REPRODUCIBILITY] ChakraNet ckpt  : /kaggle/input/datasets/gokulrocky/final-om-evlautation-upload/weights/chakra_transformer_best.pth\n[INFO] Device: cuda\n2026-09-06 12:32:38,075 [HW-MONITOR] INFO Loading pretrained weights from Hugging Face hub (timm/vit_large_patch16_384.augreg_in21k_ft_in1k)\n2026-09-06 12:32:38,516 [HW-MONITOR] INFO HTTP Request: HEAD https://huggingface.co/timm/vit_large_patch16_384.augreg_in21k_ft_in1k/resolve/main/model.safetensors \"HTTP/1.1 302 Found\"\n",
     "output_type": "stream"
    },
    {
     "name": "stderr",
     "text": "Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.\n",
     "output_type": "stream"
    },
    {
     "name": "stdout",
     "text": "2026-09-06 12:32:38,517 [HW-MONITOR] WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.\n2026-09-06 12:32:38,755 [HW-MONITOR] INFO HTTP Request: GET https://huggingface.co/api/models/timm/vit_large_patch16_384.augreg_in21k_ft_in1k/xet-read-token/1a0edc93ef27935b955ce67a344087b140f85d47 \"HTTP/1.1 200 OK\"\n",
     "output_type": "stream"
    },
    {
     "output_type": "display_data",
     "data": {
      "text/plain": "model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]",
      "application/vnd.jupyter.widget-view+json": {
       "version_major": 2,
       "version_minor": 0,
       "model_id": "c0002565dbaf45ef92c6eb5b54237421"
      }
     },
     "metadata": {}
    },
    {
     "name": "stdout",
     "text": "2026-09-06 12:32:49,501 [HW-MONITOR] INFO [timm/vit_large_patch16_384.augreg_in21k_ft_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.\n[WARN] ChakraNet: No weights found. Running with random initialization.\n[CHECKPOINT OK] All keys matched exactly.\n[FIX APPLIED] segmenter.model.eval() called - BN/Dropout now in inference mode.\nTotal valid diagnostic images: 1200\n[ASSUMPTION CHECK] Oracle crop uses 25% bbox padding, assumed to match your E2E pipeline's YOLO->crop expansion logic. If your actual pipeline uses a different padding ratio, the oracle number below is not apples-to-apples with your full-pipeline Dice of 0.8370 - verify this before drawing conclusions.\n",
     "output_type": "stream"
    },
    {
     "traceback": [
      "\u001b[0;31m---------------------------------------------------------------------------\u001b[0m",
      "\u001b[0;31merror\u001b[0m                                     Traceback (most recent call last)",
      "\u001b[0;32m/tmp/ipykernel_58/2506199719.py\u001b[0m in \u001b[0;36m<cell line: 0>\u001b[0;34m()\u001b[0m\n\u001b[1;32m    256\u001b[0m \u001b[0;34m\u001b[0m\u001b[0m\n\u001b[1;32m    257\u001b[0m \u001b[0;32mif\u001b[0m \u001b[0m__name__\u001b[0m \u001b[0;34m==\u001b[0m \u001b[0;34m\"__main__\"\u001b[0m\u001b[0;34m:\u001b[0m\u001b[0;34m\u001b[0m\u001b[0;34m\u001b[0m\u001b[0m\n\u001b[0;32m--> 258\u001b[0;31m     \u001b[0mrun_diagnostics\u001b[0m\u001b[0;34m(\u001b[0m\u001b[0;34m)\u001b[0m\u001b[0;34m\u001b[0m\u001b[0;34m\u001b[0m\u001b[0m\n\u001b[0m",
      "\u001b[0;32m/tmp/ipykernel_58/2506199719.py\u001b[0m in \u001b[0;36mrun_diagnostics\u001b[0;34m()\u001b[0m\n\u001b[1;32m    211\u001b[0m \u001b[0;34m\u001b[0m\u001b[0m\n\u001b[1;32m    212\u001b[0m             \u001b[0mprob_np\u001b[0m \u001b[0;34m=\u001b[0m \u001b[0mprob\u001b[0m\u001b[0;34m.\u001b[0m\u001b[0msqueeze\u001b[0m\u001b[0;34m(\u001b[0m\u001b[0;34m)\u001b[0m\u001b[0;34m.\u001b[0m\u001b[0mcpu\u001b[0m\u001b[0;34m(\u001b[0m\u001b[0;34m)\u001b[0m\u001b[0;34m.\u001b[0m\u001b[0mnumpy\u001b[0m\u001b[0;34m(\u001b[0m\u001b[0;34m)\u001b[0m\u001b[0;34m\u001b[0m\u001b[0;34m\u001b[0m\u001b[0m\n\u001b[0;32m--> 213\u001b[0;31m             \u001b[0mprob_resized\u001b[0m \u001b[0;34m=\u001b[0m \u001b[0mcv2\u001b[0m\u001b[0;34m.\u001b[0m\u001b[0mresize\u001b[0m\u001b[0;34m(\u001b[0m\u001b[0mprob_np\u001b[0m\u001b[0;34m,\u001b[0m \u001b[0;34m(\u001b[0m\u001b[0mpx2\u001b[0m \u001b[0;34m-\u001b[0m \u001b[0mpx1\u001b[0m\u001b[0;34m,\u001b[0m \u001b[0mpy2\u001b[0m \u001b[0;34m-\u001b[0m \u001b[0mpy1\u001b[0m\u001b[0;34m)\u001b[0m\u001b[0;34m,\u001b[0m \u001b[0minterpolation\u001b[0m\u001b[0;34m=\u001b[0m\u001b[0mcv2\u001b[0m\u001b[0;34m.\u001b[0m\u001b[0mINTER_LINEAR\u001b[0m\u001b[0;34m)\u001b[0m\u001b[0;34m\u001b[0m\u001b[0;34m\u001b[0m\u001b[0m\n\u001b[0m\u001b[1;32m    214\u001b[0m \u001b[0;34m\u001b[0m\u001b[0m\n\u001b[1;32m    215\u001b[0m             \u001b[0mc_prob\u001b[0m \u001b[0;34m=\u001b[0m \u001b[0mnp\u001b[0m\u001b[0;34m.\u001b[0m\u001b[0mzeros\u001b[0m\u001b[0;34m(\u001b[0m\u001b[0;34m(\u001b[0m\u001b[0;36m448\u001b[0m\u001b[0;34m,\u001b[0m \u001b[0;36m448\u001b[0m\u001b[0;34m)\u001b[0m\u001b[0;34m,\u001b[0m \u001b[0mdtype\u001b[0m\u001b[0;34m=\u001b[0m\u001b[0mnp\u001b[0m\u001b[0;34m.\u001b[0m\u001b[0mfloat32\u001b[0m\u001b[0;34m)\u001b[0m\u001b[0;34m\u001b[0m\u001b[0;34m\u001b[0m\u001b[0m\n",
      "\u001b[0;31merror\u001b[0m: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/resize.cpp:4086: error: (-215:Assertion failed) func != 0 in function 'resize'\n"
     ],
     "ename": "error",
     "evalue": "OpenCV(4.13.0) /io/opencv/modules/imgproc/src/resize.cpp:4086: error: (-215:Assertion failed) func != 0 in function 'resize'\n",
     "output_type": "error"
    }
   ],
   "execution_count": 19
  }
 ]
}